# RO-034 A* and heuristics: every number in the Note

The searches live in `images/searchkit.py` (the room grid of RO-009, 0.5 m cells, 8-connected moves of 0.5 m and
0.707 m with no corner cutting). This notebook runs each one and prints the numbers the Note quotes. Nothing here is
random, so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
from searchkit import *
LS, LG = (0, 0), (13, 3)          # the long trip: bottom-left corner cell to the cell at the right wall
r3 = lambda x: round(x, 3)

## 1. The grid and the two trips

In [2]:
print('blocked', int(BLOCKED.sum()), 'free', len(free_cells()))
print('start cell', START, 'goal cell', GOAL)
for j in range(NY - 1, -1, -1):
    print(''.join('S' if (i, j) == START else 'G' if (i, j) == GOAL else '#' if BLOCKED[j, i] else '.' for i in range(NX)))

blocked 36 free 76
start cell (4, 2) goal cell (7, 7)
.......G...###
.##........###
.##...#####.##
......#####.##
......#####...
....S.#####...
...........##.
..............


## 2. Dijkstra against A* on the door trip (Overview, Figure 1)

In [3]:
D, A = dijkstra(trace=True), astar(trace=True)
print('Dijkstra:', len(D['order']), 'cells taken off the open list, route', r3(D['cost']), 'm')
print('A* (octile):', len(A['order']), 'cells, route', r3(A['cost']), 'm')
dS = dist_from(START)
print('free cells nearer than 3.414 m to S:', sum(v < 3.4142 - 1e-6 for v in dS.values()))
print('A* stored at the end:', len(A['snaps'][-1]['closed']), 'closed +', len(A['snaps'][-1]['open']), 'open')

Dijkstra: 54 cells taken off the open list, route 3.414 m
A* (octile): 9 cells, route 3.414 m
free cells nearer than 3.414 m to S: 53
A* stored at the end: 9 closed + 14 open


## 3. The guess: octile and straight-line distance (Figure 2)

In [4]:
print('octile from S', r3(octile(START)), '= 3 x 0.707 + 2 x 0.5 =', r3(3 * S2 + 2 * C))
print('straight line from S', r3(straight(START)), '= 0.5 x sqrt(3^2 + 5^2) =', r3(0.5 * math.hypot(3, 5)))
Hs = dist_from(GOAL)
print('true distance from S', r3(Hs[START]))

octile from S 3.121 = 3 x 0.707 + 2 x 0.5 = 3.121
straight line from S 2.915 = 0.5 x sqrt(3^2 + 5^2) = 2.915
true distance from S 3.414


## 4. Greedy best-first on the long trip (Figure 3)

In [5]:
gr, best, dl = greedy(start=LS, goal=LG), astar(start=LS, goal=LG), dijkstra(start=LS, goal=LG)
print('greedy:', len(gr['order']), 'cells, route', r3(gr['cost']))
print('shortest (A*):', r3(best['cost']), 'in', len(best['order']), 'cells; Dijkstra', len(dl['order']), 'cells')
print('greedy on the door trip:', r3(greedy()['cost']), 'in', len(greedy()['order']), 'cells')
print('h at the long trip start', r3(octile(LS, LG)))

greedy: 21 cells, route 9.536
shortest (A*): 8.0 in 43 cells; Dijkstra 71 cells
greedy on the door trip: 3.414 in 7 cells
h at the long trip start 7.121


## 5. The A* walk-through (Figure 4)

In [6]:
for k, s in enumerate(A['snaps'], 1):
    c = s['cur']
    print(k, c, 'g', r3(s['g'][c]), 'h', r3(octile(c)), 'f', r3(s['g'][c] + octile(c)))
print('g of the cell two above S: after step 1', r3(A['snaps'][0]['g'].get((4, 4), float('nan'))), '... after step 2',
      r3(A['snaps'][1]['g'][(4, 4)]), '-> after step 5', r3(A['snaps'][4]['g'][(4, 4)]))
print('g of the cell three above S: after step 3', r3(A['snaps'][2]['g'][(4, 5)]), '-> after step 6', r3(A['snaps'][5]['g'][(4, 5)]))
print('route', A['path'])
last = A['snaps'][-1]
print('open list when G comes off:', sorted((r3(last['g'][c] + octile(c)), c) for c in last['open'])[:4])

1 (4, 2) g 0.0 h 3.121 f 3.121
2 (5, 3) g 0.707 h 2.414 f 3.121
3 (5, 4) g 1.207 h 1.914 f 3.121
4 (5, 5) g 1.707 h 1.414 f 3.121
5 (4, 3) g 0.5 h 2.621 f 3.121
6 (4, 4) g 1.0 h 2.121 f 3.121
7 (5, 6) g 2.207 h 1.207 f 3.414
8 (6, 7) g 2.914 h 0.5 f 3.414
9 (7, 7) g 3.414 h 0.0 f 3.414
g of the cell two above S: after step 1 nan ... after step 2 1.414 -> after step 5 1.0
g of the cell three above S: after step 3 1.914 -> after step 6 1.5
route [(4, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
open list when G comes off: [(3.414, (4, 5)), (3.414, (5, 2)), (3.414, (6, 6)), (3.536, (3, 3))]


## 6. Admissible: no guess above the true distance (Figure 5)

In [7]:
cells = free_cells()
print('octile > true anywhere?', any(octile(c) > Hs[c] + 1e-9 for c in cells))
print('straight > true anywhere?', any(straight(c) > Hs[c] + 1e-9 for c in cells))
print('cells where octile = true:', sum(abs(octile(c) - Hs[c]) < 1e-9 for c in cells), 'of', len(cells))
print('max true distance', r3(max(Hs.values())))
a2 = astar(h=straight)
print('A* with the straight line:', len(a2['order']), 'cells, route', r3(a2['cost']))
print('cell three above S: g', r3(last['g'][(4, 5)]), 'h', r3(octile((4, 5))), 'true left', r3(Hs[(4, 5)]))
n = (3, 2)   # the cell left of S, on the open list when G comes off (Section 5.2)
print('cell left of S: open?', n in last['open'], 'g', r3(last['g'][n]), 'h', r3(octile(n)), 'f', r3(last['g'][n] + octile(n)))

octile > true anywhere? False
straight > true anywhere? False
cells where octile = true: 31 of 76
max true distance 11.121
A* with the straight line: 11 cells, route 3.414
cell three above S: g 1.5 h 1.914 true left 1.914
cell left of S: open? True g 0.5 h 3.328 f 3.828


## 7. Consistency (Figures 6 and 7)

In [8]:
print('octile consistent on every move:', all(octile(c) <= s + octile(n) + 1e-12 for c in cells for n, s in neighbours(c)))
print('straight consistent on every move:', all(straight(c) <= s + straight(n) + 1e-12 for c in cells for n, s in neighbours(c)))
print('one move, (5,5) to (5,6):', r3(octile((5, 5))), '<=', 0.5, '+', r3(octile((5, 6))))
for a, b in RM_EDGES:
    print('roadmap edge', a, b, r3(rm_len(a, b)))
H1 = dict(S=0.0, A=rm_len('A', 'C') + rm_len('C', 'G'), B=0.0, C=0.0, G=0.0)
H2 = dict(H1, A=rm_len('A', 'C'))
for name, H in (('run 1', H1), ('run 2', H2)):
    R = rm_astar(H)
    print(name, 'h(A) =', r3(H['A']), 'order', [s['cur'] for s in R['steps']], 'route', R['path'], r3(R['cost']))
    for s in R['steps']:
        print('   took', s['cur'], 'open', {v: tuple(r3(x) for x in t) for v, t in s['open'].items()}, 'skipped', s['skipped'])

octile consistent on every move: True
straight consistent on every move: True
one move, (5,5) to (5,6): 1.414 <= 0.5 + 1.207
roadmap edge S A 1.581
roadmap edge S B 1.521
roadmap edge A C 0.906
roadmap edge B C 1.671
roadmap edge C G 1.166
run 1 h(A) = 2.072 order ['S', 'B', 'C', 'A', 'G'] route ['S', 'B', 'C', 'G'] 4.358
   took S open {'A': (1.581, 2.072, 3.653), 'B': (1.521, 0.0, 1.521)} skipped []
   took B open {'A': (1.581, 2.072, 3.653), 'C': (3.192, 0.0, 3.192)} skipped []
   took C open {'A': (1.581, 2.072, 3.653), 'G': (4.358, 0.0, 4.358)} skipped []
   took A open {'G': (4.358, 0.0, 4.358)} skipped [('C', 2.4866773438979313)]
   took G open {} skipped []
run 2 h(A) = 0.906 order ['S', 'B', 'A', 'C', 'G'] route ['S', 'A', 'C', 'G'] 3.653
   took S open {'A': (1.581, 0.906, 2.487), 'B': (1.521, 0.0, 1.521)} skipped []
   took B open {'A': (1.581, 0.906, 2.487), 'C': (3.192, 0.0, 3.192)} skipped []
   took A open {'C': (2.487, 0.0, 2.487)} skipped []
   took C open {'G': (3.653

## 8. Iterative deepening and IDA* (Figure 8)

In [9]:
def dfid(start=START, goal=GOAL):
    out = []
    for L in range(10):
        cnt, found = [0], [None]
        def dfs(path):
            c = path[-1]
            cnt[0] += 1
            if c == goal:
                found[0] = list(path); return True
            if len(path) - 1 == L:
                return False
            for n, s in neighbours(c):
                if n in path:
                    continue
                path.append(n)
                if dfs(path):
                    return True
                path.pop()
            return False
        dfs([start])
        out.append((L, cnt[0], found[0]))
        if found[0]:
            return out
runs = dfid()
for L, n, f in runs:
    print('depth limit', L, 'visits', n, 'found' if f else '')
print('total visits', sum(n for _, n, _ in runs), '; route', len(runs[-1][2]) - 1, 'moves,', r3(path_cost(runs[-1][2])), 'm')
for r in ida_star():
    print('IDA* limit', r3(r['limit']), 'visits', len(r['visits']), 'next limit', r3(r['next']), 'found', r['found'] and r3(r['found'][1]))
print('longest route held in memory:', max(len(p) for r in ida_star() for p in r['paths']), 'cells')

depth limit 0 visits 1 
depth limit 1 visits 9 
depth limit 2 visits 58 
depth limit 3 visits 316 
depth limit 4 visits 1592 
depth limit 5 visits 7539 
depth limit 6 visits 15364 found
total visits 24879 ; route 6 moves, 3.828 m
IDA* limit 3.121 visits 9 next limit 3.414 found None
IDA* limit 3.414 visits 8 next limit 3.536 found 3.414
longest route held in memory: 7 cells


## 9. Backward and bidirectional search (Figures 9 and 10)

In [10]:
bw = dijkstra(start=GOAL, goal=START)
bi = bidirectional()
print('forward', len(D['order']), 'backward', len(bw['order']), 'bidirectional', bi['expanded'], 'route', r3(bi['cost']))
print('w', bi['w'], 'route through w', r3(bi['w_cost']), 'best meeting cell', bi['x'])
print('free cells nearer than 3.414 m to G:', sum(v < 3.4142 - 1e-6 for v in Hs.values()))
tr = bidirectional((4, 4), GOAL)
print('from (4,4): stops after', tr['expanded'], 'cells; w', tr['w'], r3(tr['gf'][tr['w']]), '+', r3(tr['gb'][tr['w']]), '=', r3(tr['w_cost']))
print('   best x', tr['x'], r3(tr['gf'][tr['x']]), '+', r3(tr['gb'][tr['x']]), '=', r3(tr['cost']), '; true', r3(dist_from((4, 4))[GOAL]))

forward 54 backward 31 bidirectional 38 route 3.414
w (5, 4) route through w 3.414 best meeting cell (4, 4)
free cells nearer than 3.414 m to G: 30
from (4,4): stops after 26 cells; w (4, 6) 1.0 + 1.707 = 2.707
   best x (4, 5) 0.5 + 1.914 = 2.414 ; true 2.414


## 10. Several goals: three charging docks, a busy dock, a moving person (Figures 11 and 12)

In [11]:
DOCKS = {'G': GOAL, 'D1': (0, 7), 'D2': (10, 1)}   # under the door, top-left corner, below the table's right end
runs = {n: astar(goal=c) for n, c in DOCKS.items()}
for n, c in DOCKS.items():
    print(n, c, 'straight line', r3(straight(START, c)), 'octile', r3(octile(START, c)),
          'A* route', r3(runs[n]['cost']), 'in', len(runs[n]['order']), 'cells')
print('three separate A* runs:', sum(len(r['order']) for r in runs.values()), 'cells')
m = multi_goal(set(DOCKS.values()))
print('multi-goal A*: route', r3(m['cost']), 'to', m['path'][-2], ';', len(m['order']) - 1, 'cells, then V')
print('   same route as the separate run to D2:', m['path'][:-1] == runs['D2']['path'])
for c in m['order'][:-1]:
    hs = {n: octile(c, q) for n, q in DOCKS.items()}
    n = min(hs, key=hs.get)
    print('   took', c, 'g', r3(m['g'][c]), 'h', r3(hs[n]), 'from', n, 'f', r3(m['g'][c] + hs[n]))
busy = {DOCKS['D2']: 2.0}
b = multi_goal(set(DOCKS.values()), extra=busy)
print('D2 busy (edge 2 m): route', r3(b['cost']), 'to', b['path'][-2], ';', len(b['order']) - 1, 'cells;',
      'through D2', r3(runs['D2']['cost'] + 2))
nbr = lambda c: neighbours(c) + ([('V', busy.get(c, 0.0))] if c in DOCKS.values() else [])
b0 = best_first(goal='V', nbrs=nbr, h=lambda c, _: 0.0 if c == 'V' else min(octile(c, q) for q in DOCKS.values()))
print('   the same with the guess left without the 2 m:', r3(b0['cost']), 'in', len(b0['order']) - 1, 'cells')

walk = [(10 - k, 7) for k in range(11)]          # the person's cell from second k to second k + 1
R = moving_goal(walk)
route_cells = [s[0] for s in R['path'][:-1]]
print('moving person: meet in', route_cells[-1], 'at', r3(R['cost']), 's; route', route_cells, r3(path_cost(route_cells)), 'm')
print('   states taken off the open list:', [s if s == 'V' else (s[0], r3(s[1])) for s in R['order']])
print('   h at S:', min(octile(START, q) for q in walk) / 0.5, 's')
for k in range(3, 7):
    T = dS[walk[k]] / 0.5
    print('   person in', walk[k], 'from', k, 'to', k + 1, 's; robot there at', r3(T), 's ->',
          f'meet at {r3(max(T, k))} s' if T < k + 1 else 'too late')

G (7, 7) straight line 2.915 octile 3.121 A* route 3.414 in 9 cells
D1 (0, 7) straight line 3.202 octile 3.328 A* route 3.914 in 14 cells
D2 (10, 1) straight line 3.041 octile 3.207 A* route 3.207 in 7 cells
three separate A* runs: 30 cells
multi-goal A*: route 3.207 to (10, 1) ; 12 cells, then V
   same route as the separate run to D2: True
   took (4, 2) g 0.0 h 3.121 from G f 3.121
   took (5, 3) g 0.707 h 2.414 from G f 3.121
   took (5, 4) g 1.207 h 1.914 from G f 3.121
   took (5, 5) g 1.707 h 1.414 from G f 3.121
   took (4, 3) g 0.5 h 2.621 from G f 3.121
   took (4, 4) g 1.0 h 2.121 from G f 3.121
   took (5, 1) g 0.707 h 2.5 from D2 f 3.207
   took (6, 1) g 1.207 h 2.0 from D2 f 3.207
   took (7, 1) g 1.707 h 1.5 from D2 f 3.207
   took (8, 1) g 2.207 h 1.0 from D2 f 3.207
   took (9, 1) g 2.707 h 0.5 from D2 f 3.207
   took (10, 1) g 3.207 h 0.0 from D2 f 3.207
D2 busy (edge 2 m): route 3.414 to (7, 7) ; 12 cells; through D2 5.207
   the same with the guess left without the 

## 11. Weighted A* and ARA* on the long trip (Figure 13)

In [12]:
for e in (2.5, 1.2, 1.0):
    w = astar(start=LS, goal=LG, eps=e)
    print('weighted A* eps', e, ':', len(w['order']), 'cells, route', r3(w['cost']), '; bound', r3(e * best['cost']))
for r in ara_star((2.5, 1.2, 1.0), start=LS, goal=LG):
    print('ARA* eps', r['eps'], ':', len(r['order']), 'cells, route', r3(r['cost']))
w = astar(start=LS, goal=LG, eps=2.5, trace=True)
s = w['snaps'][-1]
n = (1, 0)
print('first open cell of the shortest route at the stop:', n, 'g', r3(s['g'][n]), 'h', round(octile(n, LG), 4),
      'f_eps', r3(s['g'][n] + 2.5 * octile(n, LG)), 'true left', r3(dist_from(LG)[n]))
print('door trip: eps 2.5 takes', len(astar(eps=2.5)['order']), 'cells, route', r3(astar(eps=2.5)['cost']))

weighted A* eps 2.5 : 22 cells, route 9.243 ; bound 20.0
weighted A* eps 1.2 : 38 cells, route 8.414 ; bound 9.6
weighted A* eps 1.0 : 43 cells, route 8.0 ; bound 8.0
ARA* eps 2.5 : 22 cells, route 9.243
ARA* eps 1.2 : 31 cells, route 8.414
ARA* eps 1.0 : 16 cells, route 8.0
first open cell of the shortest route at the stop: (1, 0) g 0.5 h 6.6213 f_eps 17.053 true left 7.5
door trip: eps 2.5 takes 7 cells, route 3.414


## 12. Jump point search and Theta* (Figures 14 and 15)

In [13]:
J = jps()
print('JPS: taken off', len(J['order']), 'cells', J['order'], 'route', r3(J['cost']))
print('cells ever on the JPS open list:', len({START} | {j['jp'] for j in J['jumps'] if j['jp']}))
for j in J['jumps']:
    print('  from', j['frm'], 'direction', j['d'], 'steps', len(j['steps']), 'jump point', j['jp'])
JL = jps(LS, LG)
print('long trip: JPS', len(JL['order']), 'cells; A*', len(best['order']))
T = theta_star()
print('Theta*:', T['path'], r3(T['cost']), '=', r3(math.hypot(1.0, 2.5)), '+ 0.5 ;', len(T['order']), 'cells')
print('straight line S to G clear?', line_of_sight(START, GOAL))
print('taut route through the corner (3.0, 3.0):', r3(math.dist((2.25, 1.25), (3.0, 3.0)) + math.dist((3.0, 3.0), (3.75, 3.75))))

JPS: taken off 5 cells [(4, 2), (5, 3), (5, 6), (6, 7), (7, 7)] route 3.414
cells ever on the JPS open list: 7
  from (4, 2) direction (-1, -1) steps 2 jump point None
  from (4, 2) direction (-1, 0) steps 4 jump point None
  from (4, 2) direction (-1, 1) steps 1 jump point (3, 3)
  from (4, 2) direction (0, -1) steps 2 jump point None
  from (4, 2) direction (0, 1) steps 5 jump point None
  from (4, 2) direction (1, -1) steps 2 jump point (6, 0)
  from (4, 2) direction (1, 0) steps 1 jump point None
  from (4, 2) direction (1, 1) steps 1 jump point (5, 3)
  from (5, 3) direction (1, 0) steps 0 jump point None
  from (5, 3) direction (0, 1) steps 3 jump point (5, 6)
  from (5, 3) direction (1, 1) steps 0 jump point None
  from (5, 6) direction (0, 1) steps 1 jump point None
  from (5, 6) direction (1, 0) steps 5 jump point None
  from (5, 6) direction (1, 1) steps 1 jump point (6, 7)
  from (6, 7) direction (1, 0) steps 1 jump point (7, 7)
  from (6, 7) direction (0, 1) steps 0 jump po

Theta*: [(4, 2), (6, 7), (7, 7)] 3.193 = 2.693 + 0.5 ; 10 cells
straight line S to G clear? False
taut route through the corner (3.0, 3.0): 2.965
